## **?? ??**

[???: ???? ??? ?? ??? ?? ??]
- http://www.manythings.org/anki/ ?? ??? ?? ????? ??.

- ? ???? ????? ??? ???? ?? ???-?-??? (seq2seq) ??? ??????. ?? ? ???? ???-?-??? ??? ???? ??? ???? ??? ?????.

- ? ????? ??? ?? ?? ??? ???? *"?todavia estan en casa?"*? ?? ??? ??? ???? ? *"are you still at home?"*?? ??? ??? ??? ?? ? ?? ????

- ??? ??? ??? ???? ??? ?? ??? ? ??? ?? ?? ?? ???? ????? ??? ????:

<img src="https://tensorflow.org/images/spanish-english.png" alt="spanish-english attention plot">


In [ ]:
import tensorflow as tf

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.model_selection import train_test_split

import unicodedata
import re
import numpy as np
import os
import io
import time

### ???? ??

http://www.manythings.org/anki/ ?? ??? ?? ????? ??? ????. ? ????? ?? ??? ?? ??? ?? ???? ???? ????:

```
May I borrow this book?	?Puedo tomar prestado este libro?
```

??? ??? ??????? ? ????? ??-??? ????? ??? ????. ??? ??? ? ????? ???? Google Cloud?? ??? ??? ?? ?????? ???. ????? ????? ?? ???? ????? ??? ??? ?????.

1. ? ??? *start*? *end* ??? ?????.
2. ?? ??? ?????? ??? ?????.
3. ?? ???? ???(ID) ???? ?????. (?? ? ???(ID), ???(ID) ? ??? ??? ????).
4. ? ??? ???? ?? ???? ??(padding)? ?????.

In [ ]:
# ??? ???????.
path_to_zip = tf.keras.utils.get_file(
    'spa-eng.zip', origin='http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip',
    extract=True)

path_to_file = os.path.dirname(path_to_zip)+"/spa-eng/spa.txt"

In [ ]:
# ???? ??? ??? ?? ??? ?????.
def unicode_to_ascii(s):
  return ''.join(c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn')


def preprocess_sentence(w):
  w = unicode_to_ascii(w.lower().strip())

  # ??? ?? ?? ?? ???(.)??? ??? ?????.
  # ??: "he is a boy." => "he is a boy ."
  # ??:- https://stackoverflow.com/questions/3645931/python-padding-punctuation-with-white-spaces-keeping-punctuation
  w = re.sub(r"([?.!,?])", r" \1 ", w)
  w = re.sub(r'[" "]+', " ", w)

  # (a-z, A-Z, ".", "?", "!", ",")? ??? ?? ?? ???? ?????.
  w = re.sub(r"[^a-zA-Z?.!,?]+", " ", w)

  w = w.strip()

  # ??? ??? ????? ??? ?? ?? ?? ???
  # ??? start? end ??? ?????.
  w = '<start> ' + w + ' <end>'
  return w

In [ ]:
en_sentence = u"May I borrow this book?"
sp_sentence = u"?Puedo tomar prestado este libro?"
print(preprocess_sentence(en_sentence))
print(preprocess_sentence(sp_sentence).encode('utf-8'))

In [ ]:
# 1. ??? ?? ??? ?????.
# 2. ???? ??? ???? ??? ?????.
# 3. ??? ?? ???? ??? ?? ?????: [??, ????]
def create_dataset(path, num_examples):
  lines = io.open(path, encoding='UTF-8').read().strip().split('\n')

  word_pairs = [[preprocess_sentence(w) for w in l.split('\t')]  for l in lines[:num_examples]]

  return zip(*word_pairs)

In [ ]:
en, sp = create_dataset(path_to_file, None)
print(en[-1])
print(sp[-1])

In [ ]:
def tokenize(lang):
  lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(
      filters='')
  lang_tokenizer.fit_on_texts(lang)

  tensor = lang_tokenizer.texts_to_sequences(lang)

  tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor,
                                                         padding='post')

  return tensor, lang_tokenizer

In [ ]:
def load_dataset(path, num_examples=None):
  # ???? ?? ??? ?? ?? ?? ?????.
  targ_lang, inp_lang = create_dataset(path, num_examples)

  input_tensor, inp_lang_tokenizer = tokenize(inp_lang)
  target_tensor, targ_lang_tokenizer = tokenize(targ_lang)

  return input_tensor, target_tensor, inp_lang_tokenizer, targ_lang_tokenizer

In [ ]:
# ?? ????? ??? ??? ???? ??? ??? ?????.
num_examples = 30000
input_tensor, target_tensor, inp_lang, targ_lang = load_dataset(path_to_file, num_examples)

# ?? ??? ?? ??? ?? ??? ?????.
max_length_targ, max_length_inp = target_tensor.shape[1], input_tensor.shape[1]

In [ ]:
# ?? ??? ?? ??? 80? 20?? ?????.
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

# ?? ??? ?? ??? ??? ??? ?????.
print(len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val))


In [ ]:
def convert(lang, tensor):
  for t in tensor:
    if t!=0:
      print ("%d ----> %s" % (t, lang.index_word[t]))

In [ ]:
print ("Input Language; index to word mapping")
convert(inp_lang, input_tensor_train[0])
print ()
print ("Target Language; index to word mapping")
convert(targ_lang, target_tensor_train[0])

### tf.data ???? ????

In [ ]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64
steps_per_epoch = len(input_tensor_train)//BATCH_SIZE
embedding_dim = 256
units = 1024
vocab_inp_size = len(inp_lang.word_index)+1
vocab_tar_size = len(targ_lang.word_index)+1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

In [ ]:
example_input_batch, example_target_batch = next(iter(dataset))
example_input_batch.shape, example_target_batch.shape

## ??? ??? ??? ?? ??

???(attention)? ?? ???-??? ??? ?????. ???(attention)? TensorFlow [Neural Machine Translation (seq2seq) tutorial](https://github.com/tensorflow/nmt)?? ?? ? ????. ? ??? ? ??? API ??? ?????. ? ???? seq2seq ??????? [??? ???](https://github.com/tensorflow/nmt#background-on-the-attention-mechanism)? ?????. ??? ?????? ??? ?? ??? ??? ????? ?? ???? ??? ?????. ??? ??? ????? ???? ???? ?? ??? ???? ?? ?????. ??? ??? ??? [Luong's paper](https://arxiv.org/abs/1508.04025v5)?? ?? ??? ????? ?????.

<img src="https://www.tensorflow.org/images/seq2seq/attention_mechanism.jpg" width="500" alt="attention mechanism">

??? *(batch_size, max_length, hidden_size)*? ??? ???? ??? ??? *(batch_size, hidden_size)*??? ???? ??? ?? ??(hidden state)? ???? ??? ??? ?? ?????.

??? ??? ??? ??? ???? ??? ???:

<img src="https://www.tensorflow.org/images/seq2seq/attention_equation_0.jpg" alt="attention equation 0" width="800">
<img src="https://www.tensorflow.org/images/seq2seq/attention_equation_1.jpg" alt="attention equation 1" width="800">

? ????? ???? ?? [Bahdanau ???](https://arxiv.org/pdf/1409.0473.pdf)? ?????. ???? ??? ?? ?? ???? ??? ?? ?????:

* FC = ?? ??(Dense)?
* EO = ??? ??
* H = ?? ??(hidden state)
* X = ???? ?? ??

??? ??? ???????:

* `???(score)? FC(tanh(FC(EO) + FC(H)))`? ?????.
* `??? ???? softmax(score, axis = 1)`? ?????. ????? ?????? ??? ?? ????? ???(score)? ??? *(batch_size, max_length, hidden_size)*?? ??? *??? ?*? ?????. `Max_length`? ??? ?????. ??? ??? ???? ????? ???? ??? ?????? ? ?? ??? ? ????.
* `???? ??(context vector)? sum(??? ??? * EO, axis = 1)`? ?????. ?? ?? ??? ??? ?? ?????.
* `??? ??(embedding output)`? ??? X? ?? ??? ????? ??? ?????.
* `??? ??(merged vector)? concat(??? ??, ???? ??(context vector))`? ????.
* ?? ?? ??? ??? GRU? ?????.

? ???? ?? ??? ??? ??? ??? ???? ????:

In [ ]:
class Encoder(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
    super(Encoder, self).__init__()
    self.batch_sz = batch_sz
    self.enc_units = enc_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.enc_units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')

  def call(self, x, hidden):
    x = self.embedding(x)
    output, state = self.gru(x, initial_state = hidden)
    return output, state

  def initialize_hidden_state(self):
    return tf.zeros((self.batch_sz, self.enc_units))

In [ ]:
encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)

# ?? ??
sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)
print ('Encoder output shape: (batch size, sequence length, units) {}'.format(sample_output.shape))
print ('Encoder Hidden state shape: (batch size, units) {}'.format(sample_hidden.shape))

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
  def __init__(self, units):
    super(BahdanauAttention, self).__init__()
    self.W1 = tf.keras.layers.Dense(units)
    self.W2 = tf.keras.layers.Dense(units)
    self.V = tf.keras.layers.Dense(1)

  def call(self, query, values):
    # ?? ?? ??(query hidden state)? (batch_size, hidden size)??? ???? ????.
    # query_with_time_axis? (batch_size, 1, hidden size)??? ???? ????.
    # values? (batch_size, max_len, hidden size)??? ???? ????.
    # ???(score)??? ?? ??? ????? ?? ?? ???? ??? ??? ?????.
    query_with_time_axis = tf.expand_dims(query, 1)

    # score? (batch_size, max_length, 1)??? ???? ????.
    # score? self.V? ???? ??? ??? ?? 1? ????.
    # self.V? ???? ?? ??? (batch_size, max_length, units)??? ???? ????.
    score = self.V(tf.nn.tanh(
        self.W1(query_with_time_axis) + self.W2(values)))

    # attention_weights? (batch_size, max_length, 1)??? ???? ????.
    attention_weights = tf.nn.softmax(score, axis=1)

    # ???? ???? ??(context_vector)? (batch_size, hidden_size)??? ???? ????.
    context_vector = attention_weights * values
    context_vector = tf.reduce_sum(context_vector, axis=1)

    return context_vector, attention_weights

In [ ]:
attention_layer = BahdanauAttention(10)
attention_result, attention_weights = attention_layer(sample_hidden, sample_output)

print("Attention result shape: (batch size, units) {}".format(attention_result.shape))
print("Attention weights shape: (batch_size, sequence_length, 1) {}".format(attention_weights.shape))

In [ ]:
class Decoder(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
    super(Decoder, self).__init__()
    self.batch_sz = batch_sz
    self.dec_units = dec_units
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(self.dec_units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')
    self.fc = tf.keras.layers.Dense(vocab_size)

    # ???? ?????.
    self.attention = BahdanauAttention(self.dec_units)

  def call(self, x, hidden, enc_output):
    # enc_output? (batch_size, max_length, hidden_size)??? ???? ????.
    context_vector, attention_weights = self.attention(hidden, enc_output)

    # ????? ??? ? x? (batch_size, 1, embedding_dim)??? ???? ????.
    x = self.embedding(x)

    # ???? ??? ??? ??? ??? ?? x? ??? (batch_size, 1, embedding_dim + hidden_size)??? ???? ????.
    x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

    # ??? ??? ??? GRU? ?????.
    output, state = self.gru(x)

    # output? (batch_size * 1, hidden_size)??? ???? ????.
    output = tf.reshape(output, (-1, output.shape[2]))

    # output? (batch_size, vocab)??? ???? ????.
    x = self.fc(output)

    return x, state, attention_weights

In [ ]:
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

sample_decoder_output, _, _ = decoder(tf.random.uniform((BATCH_SIZE, 1)),
                                      sample_hidden, sample_output)

print ('Decoder output shape: (batch_size, vocab size) {}'.format(sample_decoder_output.shape))

## ??? ??? ?? ?? ????

In [ ]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
  mask = tf.math.logical_not(tf.math.equal(real, 0))
  loss_ = loss_object(real, pred)

  mask = tf.cast(mask, dtype=loss_.dtype)
  loss_ *= mask

  return tf.reduce_mean(loss_)

## ????? (?? ?? ??)

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 encoder=encoder,
                                 decoder=decoder)

## ?? ?? ????

1. *??? ??*? *??? ?? ??(hidden state)*? ???? *???*? ??? *??*? ?????.
2. ??? ??, ??? ?? ??(hidden state), ??? ?? (*start ??*)? ???? ?????.
3. ?? ?? ?? ?? ???? *?? ?*? *??? ?? ??(hidden state)*? ?????.
4. ? ??? ??? ?? ??(hidden state)? ?? ??? ???? ?? ?? ???? ??? ?????.
5. ???? ?? ?? ??? ???? ??? *?? ??(teacher forcing)*? ?????.
6. *?? ??(teacher forcing)*? *?? ??*? ???? *?? ??*?? ???? ?? ?????.
7. ??? ??? ?????(gradients)? ???? ?? ?????(optimizer)? ???(backpropagate)? ?????.

In [ ]:
@tf.function
def train_step(inp, targ, enc_hidden):
  loss = 0

  with tf.GradientTape() as tape:
    enc_output, enc_hidden = encoder(inp, enc_hidden)

    dec_hidden = enc_hidden

    dec_input = tf.expand_dims([targ_lang.word_index['<start>']] * BATCH_SIZE, 1)

    # ?? ??(teacher forcing) - ?? ???? ??? ??(feeding)???.
    for t in range(1, targ.shape[1]):
      # enc_output? ???? ?????.
      predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)

      loss += loss_function(targ[:, t], predictions)

      # ?? ??(teacher forcing)? ?????.
      dec_input = tf.expand_dims(targ[:, t], 1)

  batch_loss = (loss / int(targ.shape[1]))

  variables = encoder.trainable_variables + decoder.trainable_variables

  gradients = tape.gradient(loss, variables)

  optimizer.apply_gradients(zip(gradients, variables))

  return batch_loss

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
  start = time.time()

  enc_hidden = encoder.initialize_hidden_state()
  total_loss = 0

  for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
    batch_loss = train_step(inp, targ, enc_hidden)
    total_loss += batch_loss

    if batch % 100 == 0:
      print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,
                                                   batch,
                                                   batch_loss.numpy()))
  # ???? 2? ?????? ?? ?? (?????)
  if (epoch + 1) % 2 == 0:
    checkpoint.save(file_prefix = checkpoint_prefix)

  print('Epoch {} Loss {:.4f}'.format(epoch + 1,
                                      total_loss / steps_per_epoch))
  print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

## ??? ??? ????

* ?? ??? ??? *?? ??(teacher forcing)*? ???? ??? ?? ????? ?? ??? ?????. ? ??? ??(time step)?? ?? ??? ???? ??? ?? ??(hidden state)? ?? ?? ?? ???? ?????.
* ??? *end ??? ??? ? ???? ?? ?????. *.
* ??? *? ??? ??(time step)? ?? ??? ???*? ?????.

??: ??? ??? ??? ??? ?? ? ? ?? ?????.

In [ ]:
def evaluate(sentence):
  attention_plot = np.zeros((max_length_targ, max_length_inp))

  sentence = preprocess_sentence(sentence)

  inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
  inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],
                                                         maxlen=max_length_inp,
                                                         padding='post')
  inputs = tf.convert_to_tensor(inputs)

  result = ''

  hidden = [tf.zeros((1, units))]
  enc_out, enc_hidden = encoder(inputs, hidden)

  dec_hidden = enc_hidden
  dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)

  for t in range(max_length_targ):
    predictions, dec_hidden, attention_weights = decoder(dec_input,
                                                         dec_hidden,
                                                         enc_out)

    # ??? ??? ???? ????? ?? ??? ???? ?????.
    attention_weights = tf.reshape(attention_weights, (-1, ))
    attention_plot[t] = attention_weights.numpy()

    predicted_id = tf.argmax(predictions[0]).numpy()

    result += targ_lang.index_word[predicted_id] + ' '

    if targ_lang.index_word[predicted_id] == '<end>':
      return result, sentence, attention_plot

    # ??? ID? ??? ?? ?????.
    dec_input = tf.expand_dims([predicted_id], 0)

  return result, sentence, attention_plot

In [ ]:
# ??? ???? ??? ?? ?????.
def plot_attention(attention, sentence, predicted_sentence):
  fig = plt.figure(figsize=(10,10))
  ax = fig.add_subplot(1, 1, 1)
  ax.matshow(attention, cmap='viridis')

  fontdict = {'fontsize': 14}

  ax.set_xticklabels([''] + sentence, fontdict=fontdict, rotation=90)
  ax.set_yticklabels([''] + predicted_sentence, fontdict=fontdict)

  ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
  ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

  plt.show()

In [ ]:
def translate(sentence):
  result, sentence, attention_plot = evaluate(sentence)

  print('Input: %s' % (sentence))
  print('Predicted translation: {}'.format(result))

  attention_plot = attention_plot[:len(result.split(' ')), :len(sentence.split(' '))]
  plot_attention(attention_plot, sentence.split(' '), result.split(' '))

## ??? ?????(checkpoint)? ???? ?????

In [ ]:
# checkpoint_dir?? ?? ?? ?????(checkpoint)? ?????.
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
translate('hace mucho frio aqui.')

In [ ]:
translate('esta es mi vida.')

In [ ]:
translate('?todavia estan en casa?')

In [ ]:
# ??? ??
translate(u'trata de averiguarlo.')

In [ ]:
# ??? ??
translate('vamos a cenar ma?ana por la ma?ana')

In [ ]:
# ??? ??
translate('vayamos a paris y pasemos un buen rato')